# LifeMate 2026: Run Baseline Qwen3-4B on Dev Set (300 samples)

**Mục tiêu:** Chạy đánh giá Baseline của Planner trên tập 300 mẫu dev (`eval/splits/dev_ids.txt`) sử dụng GPU miễn phí trên Kaggle (GPU T4 x2 hoặc GPU P100 16GB VRAM).

### Yêu cầu cài đặt trên Kaggle:
1. **Settings** -> **Accelerator**: Chọn **GPU T4 x2** hoặc **GPU P100**.
2. **Settings** -> **Internet**: Bật **ON** (để tải thư viện và model từ HuggingFace).
3. **Data**: Thêm Dataset chứa thư mục `train` và `public_test` của BTC.

In [ ]:
# 1. Cài đặt các thư viện cần thiết
!pip install -q -U vllm transformers peft accelerate datasets chunkformer

In [ ]:
# 2. Clone repository ZAIC_2026 và chuyển sang nhánh planner/vuong
import os
if not os.path.exists("ZAIC_2026"):
    !git clone https://github.com/Bamboohoccode/ZAIC_2026.git
%cd ZAIC_2026
!git checkout planner/vuong
!git pull origin planner/vuong

In [ ]:
# 3. Thiết lập đường dẫn dữ liệu data/
import os, shutil
from pathlib import Path

os.makedirs("data", exist_ok=True)

# Tự động tìm thư mục data trong /kaggle/input
kaggle_input = Path("/kaggle/input")
found_train = list(kaggle_input.rglob("train.jsonl"))
if found_train:
    src_train_dir = found_train[0].parent
    src_root = src_train_dir.parent
    print(f"Tìm thấy dữ liệu tại: {src_root}")
    if not os.path.exists("data/train"):
        os.symlink(str(src_train_dir), "data/train")
    pub_dirs = list(kaggle_input.rglob("public_test.jsonl"))
    if pub_dirs and not os.path.exists("data/public_test"):
        os.symlink(str(pub_dirs[0].parent), "data/public_test")
    print("Đã liên kết dữ liệu vào data/:")
    !ls data
else:
    print("LƯU Ý: Vui lòng upload dataset train+public_test vào Kaggle và kiểm tra lại đường dẫn!")

In [ ]:
# 4. Chạy Baseline trên 300 mẫu dev bằng Qwen3-4B (chế độ --text, không mapper)
# Lưu ý: Thêm --gpu-mem 0.90 và --max-ctx 8192 để vừa vặn KV cache trên GPU T4 16GB của Kaggle
os.makedirs("runs", exist_ok=True)

!python -m lifemate.pipeline \
    --root data \
    --split train \
    --text \
    --llm Qwen/Qwen3-4B-Instruct-2507 \
    --mapper none \
    --gpu-mem 0.90 \
    --max-ctx 8192 \
    --ids eval/splits/dev_ids.txt \
    --out runs/dev_baseline.jsonl

In [ ]:
# 5. Chấm điểm và hiển thị báo cáo chẩn đoán lỗi chi tiết
!python eval/scorer.py \
    --root data \
    --split train \
    --sub runs/dev_baseline.jsonl \
    --dev-ids eval/splits/dev_ids.txt

In [ ]:
# 6. Đóng gói kết quả để tải về máy hoặc commit lên repo
!zip -r /kaggle/working/dev_baseline_results.zip runs/dev_baseline.jsonl*
print("Hoàn tất! Bạn có thể tải file dev_baseline_results.zip từ mục Output của Kaggle.")